In [ ]:
import os

os.environ["SPS_HOME"] = "/Users/z5114326/Documents/GitHub/python-fsps/src/fsps/libfsps"

import fsps
import gc_utils
import h5py
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

In [ ]:
sim_lst = ["m12b", "m12c", "m12f", "m12m"]
# sim_lst = ["m12i"]
sim_dir = "/Users/z5114326/Documents/simulations/"

band_type = "JC"  # "SDSS" or "JC"
# bands = ["sdss_g", "sdss_z"]
bands = ["v"]

snap = 600
min_mass = 1e4

In [ ]:
compute_vega_mags = band_type == "JC"

sp = fsps.StellarPopulation(
    imf_type=2,
    zcontinuous=1,
    compute_vega_mags=compute_vega_mags,
)

sp.params["add_dust_emission"] = False
sp.params["add_neb_continuum"] = False
sp.params["add_neb_emission"] = False
sp.params["add_agb_dust_model"] = False
sp.params["dust1"] = 0.0
sp.params["dust2"] = 0.0
sp.params["fcstar"] = 0.0
sp.params["agb_dust"] = 0.0
sp.params["tpagb_norm_type"] = 0

In [ ]:
public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
timez0 = float(pub_data[pub_data["index"] == snap]["time_Gyr"].values[0])
pub_snaps = np.array(pub_data["index"])

all_data_fil = sim_dir + "/" + "m12i" + "/" + "m12i" + "_res7100/" + "snapshot_times.txt"
all_data = pd.read_table(all_data_fil, comment="#", header=None, sep=r"\s+")
all_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
all_times = np.array(all_data["time_Gyr"])
all_snaps = np.array(all_data["index"])

snap_times = {
    gc_utils.snapshot_name(s): float(pub_data.loc[pub_data["index"] == s, "time_Gyr"].values[0])
    for s in pub_snaps
}

In [ ]:
sim_dict = {}

# ------------------------------------------------
# precompute snapshot times once
# ------------------------------------------------
snap_times = {
    gc_utils.snapshot_name(s): float(pub_data.loc[pub_data["index"] == s, "time_Gyr"].values[0])
    for s in pub_snaps
}

timez0 = snap_times[gc_utils.snapshot_name(snap)]

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    sim_dict[sim] = {}

    with h5py.File(ghost_file, "r") as ghost_data:
        for it_id, it_grp in ghost_data.items():
            src = it_grp["source"]

            # ---------------------------
            # read once
            # ---------------------------
            grpid = src["grpid"][()]
            amsk = grpid == 0

            logm_tfor = src["logm_tfor"][()]
            logm_tz0 = src["logm_tz0"][()]

            m_tfo = 10.0**logm_tfor
            m_tfo_ev = 0.55 * m_tfo  # stellar-evolution corrected mass

            m_tz0 = np.zeros_like(logm_tz0, dtype=float)
            msk = logm_tz0 != -1
            m_tz0[msk] = 10.0 ** logm_tz0[msk]

            tfor = src["tfor"][()]
            tdis = src["tdis"][()]
            tacc = src["tacc"][()]
            feh = src["feh"][()]
            gcids = src["gcid"][()]

            tdis = np.where(tdis == -1, np.inf, tdis)

            rz0 = np.linalg.norm(src["pxyz_snap600"][()], axis=1)

            # ---------------------------
            # vectorized ages
            # ---------------------------
            age = timez0 - tfor
            age_p = np.where(np.isfinite(tdis), tdis - tfor, age)

            t_tz0 = src["torb_600"][()]
            s_tz0 = src["s_flag"][()].astype(bool)
            sa_flag = src["sa_flag"][()].astype(bool)

            # ---------------------------
            # snapshot mass table
            #   + tfor (m_tfo_ev)
            #   + tdis (0)
            # ---------------------------
            n_gc = len(gcids)
            n_snap = len(pub_snaps)

            mass_arr = np.full((n_gc, n_snap + 2), np.nan)
            time_arr = np.full((n_gc, n_snap + 2), np.nan)

            # ---- formation point ----
            mass_arr[:, 0] = m_tfo_ev
            time_arr[:, 0] = tfor

            # ---- simulation snapshots ----
            for j, (snap_id, time) in enumerate(snap_times.items(), start=1):
                snp = it_grp["snapshots"][snap_id]
                snp_gcids = snp["gcid"][()]
                snp_mass = 10.0 ** snp["logm"][()]

                common, idx_snp, idx_src = np.intersect1d(snp_gcids, gcids, return_indices=True)

                mass_arr[idx_src, j] = snp_mass[idx_snp]
                time_arr[idx_src, j] = time

                # ---- disruption point (only if disrupted) ----
                has_disrupted = np.isfinite(tdis)

                mass_arr[has_disrupted, -1] = 0.0
                time_arr[has_disrupted, -1] = tdis[has_disrupted]

            # ---------------------------
            # final mass cut
            # ---------------------------
            m_tfo_msk = m_tfo >= min_mass

            sim_dict[sim][it_id] = {
                "gcid": gcids[m_tfo_msk],
                "feh": feh[m_tfo_msk],
                "tfor": tfor[m_tfo_msk],
                "tdis": tdis[m_tfo_msk],
                "tacc": tacc[m_tfo_msk],
                "age": age[m_tfo_msk],
                "age_p": age_p[m_tfo_msk],
                "rz0": rz0[m_tfo_msk],
                "t_tz0": t_tz0[m_tfo_msk],
                "s_tz0": s_tz0[m_tfo_msk],
                "sa_flag": sa_flag[m_tfo_msk],
                "acc": amsk[m_tfo_msk],
                # ---- masses ----
                "m_tfo": m_tfo[m_tfo_msk],
                "m_tfo_ev": m_tfo_ev[m_tfo_msk],
                "m_tz0": m_tz0[m_tfo_msk],
                # ---- time-resolved mass history ----
                "m_snp": mass_arr[m_tfo_msk],
                "t_snp": time_arr[m_tfo_msk],
            }

In [ ]:
def build_mass_interpolators(dat):
    interps = []

    for m, t in zip(dat["m_snp"], dat["t_snp"]):
        msk = np.isfinite(m) & (m > 0)
        if np.sum(msk) < 2:
            interps.append(None)
            continue

        order = np.argsort(t[msk])
        ts = t[msk][order]
        ms = m[msk][order]

        interps.append(interp1d(ts, ms, kind="linear", bounds_error=False, fill_value=(ms[0], 0.0)))

    return interps


for sim in sim_lst:
    for it_id in sim_dict[sim]:
        sim_dict[sim][it_id]["mass_interp"] = build_mass_interpolators(sim_dict[sim][it_id])

In [ ]:
# ============================================================
# BUILD TIME-DEPENDENT GC MAGNITUDES INTO ssp_dict
# ============================================================

dt = 0.2
tims = np.arange(0.4, timez0, dt)
tims = np.append(tims, timez0)

print("\n================ BUILDING ssp_dict ================\n")

bands_tag = "_".join(bands)
n_band = len(bands)

ssp_dict = {}

for sim in sim_lst:
    print(f"[SIM] {sim}")

    # --------------------------------------------------------
    # Collect (age, feh) pairs needed for THIS SIM
    # --------------------------------------------------------
    ages_all = []
    fehs_all = []

    for it_id, dat in sim_dict[sim].items():
        tfor = dat["tfor"]
        feh = dat["feh"]

        for t in tims:
            age_t = t - tfor
            msk = age_t > 0
            if np.any(msk):
                ages_all.append(age_t[msk])
                fehs_all.append(feh[msk])

    ages_all = np.concatenate(ages_all)
    fehs_all = np.concatenate(fehs_all)

    ssp_params = np.column_stack((ages_all, fehs_all))
    unique_ssps = np.unique(ssp_params, axis=0)

    print(f"Unique SSPs needed: {len(unique_ssps)}")

    # --------------------------------------------------------
    # FSPS cache (per sim, per band system)
    # --------------------------------------------------------
    cache_dir = f"../data/ssp_caches/{sim}"
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = f"{cache_dir}/ssp_{band_type}_{bands_tag}.npz"

    if os.path.exists(cache_path):
        print("  → Loading FSPS cache")
        cache = np.load(cache_path)
        ssp_mag = cache["ssp_mag"]
        ssp_mass = cache["ssp_mass"]
        cached_ssps = cache["unique_ssps"]

        if not np.array_equal(cached_ssps, unique_ssps):
            print("SSP grid mismatch — recomputing")
            recompute = True
        else:
            recompute = False
    else:
        recompute = True

    if recompute:
        print("Computing FSPS SSP grid")
        n_ssp = len(unique_ssps)

        ssp_mag = np.empty((n_ssp, n_band))
        ssp_mass = np.empty(n_ssp)

        for i, (age, feh) in enumerate(unique_ssps):
            if i % 100 == 0 or i == n_ssp - 1:
                print(f"    SSP {i + 1}/{n_ssp}")

            sp.params["logzsol"] = feh
            ssp_mag[i] = sp.get_mags(tage=age, bands=bands)
            ssp_mass[i] = sp.stellar_mass

        np.savez(
            cache_path,
            unique_ssps=unique_ssps,
            ssp_mag=ssp_mag,
            ssp_mass=ssp_mass,
        )

    # --------------------------------------------------------
    # Build fast lookup: (age, feh) -> SSP index
    # --------------------------------------------------------
    ssp_lookup = {(unique_ssps[i, 0], unique_ssps[i, 1]): i for i in range(len(unique_ssps))}

    # --------------------------------------------------------
    # Allocate output dictionary for this sim
    # --------------------------------------------------------
    ssp_dict[sim] = {"times": np.asarray(tims)}

    # --------------------------------------------------------
    # Compute magnitudes for each it_id
    # --------------------------------------------------------
    for it_id, dat in sim_dict[sim].items():
        print(f"    [ITER] {it_id}")

        gcids = dat["gcid"]
        tfor = dat["tfor"]
        feh = dat["feh"]
        tdis = dat["tdis"]
        tacc = dat["tacc"]
        mass_interp = dat["mass_interp"]
        insitu = dat["acc"]
        tacc = dat["tacc"]

        n_gc = len(gcids)
        n_t = len(tims)

        mags = np.full((n_gc, n_t, n_band), np.nan, dtype=float)

        for ti, t in enumerate(tims):
            alive = (tfor <= t) & (t < tdis) & (tacc <= t)

            for i in np.where(alive)[0]:
                fmass = mass_interp[i]
                if fmass is None:
                    continue

                m_now = float(fmass(t))
                if m_now <= 0:
                    continue

                age = t - tfor[i]
                if age <= 0:
                    continue

                ssp_idx = ssp_lookup.get((age, feh[i]))
                if ssp_idx is None:
                    continue

                mags[i, ti, :] = ssp_mag[ssp_idx] - 2.5 * np.log10(m_now / ssp_mass[ssp_idx])

        ssp_dict[sim][it_id] = {
            "gcid": gcids,
            "mags": mags,  # (N_gc, N_time, N_band)
            "insitu": insitu,
            "tacc": tacc,
        }

print("\n================ ssp_dict COMPLETE ================\n")